# AlphaCluster v2 — Colab Training (VS Code Plugin)

Train the AlphaCluster RL agent on Google Colab GPU via the VS Code Colab extension.

**Note:** The notebook is edited locally in VS Code but the kernel runs on Colab's remote runtime.
Source code and data are read from Google Drive.

## What's new in v2

- **CNN+Transformer** feature extractor (replaces pure CNN)
- **19-feature market observations** (OHLCV + 14 technical indicators)
- **12-feature account state** (7 original + 5 trade-tracking features)
- **8-component reward** (asymmetric PnL, fee penalty, trend-based inactivity, position management with time bonus, trade completion, churn penalty, drawdown, direction diversity bonus)
- **Curriculum learning** (3 phases: learn mechanics with partial penalties → introduce costs → full discipline with diversity annealed to 0)
- **Parallel environments** (4x SubprocVecEnv + reward normalization)
- **36-action space** — 3 directions x 4 sizes [2%, 5%, 10%, 15%] x 3 leverage levels [5x, 10x, 15x]
- **Position modification** — change leverage/size within same direction at reduced fees (delta-only)
- **Isolated margin** — liquidation loses only the allocated margin, not the entire account; episode continues

## Setup

Upload to Google Drive (`My Drive/alphacluster/`):
1. `src/` — the project source directory (copy from your local repo)
2. `data/btcusdt_5m.parquet` — candle data
3. `data/btcusdt_funding.parquet` — funding rates (optional)

Trained models will be saved back to `My Drive/alphacluster/models/`.

## Cell 1 — Mount Drive & Install Dependencies

In [ ]:
import os
import shutil
import subprocess
import sys
import warnings
from pathlib import Path

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*Gym has been unmaintained.*")

# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/alphacluster")
DRIVE_SRC = DRIVE_ROOT / "src"

assert DRIVE_SRC.exists(), (
    f"Source not found at {DRIVE_SRC}\n"
    f"Copy your local src/ directory to Google Drive: My Drive/alphacluster/src/"
)

# Copy source to local runtime storage to avoid Drive disconnection issues
LOCAL_SRC = Path("/content/src")
if LOCAL_SRC.exists():
    shutil.rmtree(LOCAL_SRC)
shutil.copytree(DRIVE_SRC, LOCAL_SRC)
print(f"Copied source to {LOCAL_SRC}")

# Install dependencies
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "stable-baselines3>=2.4,<3.0", "gymnasium>=1.0,<2.0",
    "pyarrow", "python-dotenv", "tqdm", "rich"], check=True)

# Add local copy to path
PROJECT_ROOT = DRIVE_ROOT
sys.path.insert(0, str(LOCAL_SRC))

import alphacluster
print(f"AlphaCluster loaded from {Path(alphacluster.__file__).parent}")

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. Enable GPU runtime: Runtime > Change runtime type > T4 GPU")

## Cell 2 — Load & Split Data

In [ ]:
import pandas as pd

DATA_DIR = Path("/content/drive/MyDrive/alphacluster/data")
KLINES_PATH = DATA_DIR / "btcusdt_5m.parquet"
FUNDING_PATH = DATA_DIR / "btcusdt_funding.parquet"

assert KLINES_PATH.exists(), (
    f"Kline data not found at {KLINES_PATH}\n"
    f"Upload btcusdt_5m.parquet to Google Drive: My Drive/alphacluster/data/"
)

klines_df = pd.read_parquet(KLINES_PATH, engine="pyarrow")
print(f"Loaded {len(klines_df):,} candles")
print(f"Date range: {klines_df.iloc[0, 0]} to {klines_df.iloc[-1, 0]}")

funding_df = None
if FUNDING_PATH.exists():
    funding_df = pd.read_parquet(FUNDING_PATH, engine="pyarrow")
    print(f"Loaded {len(funding_df):,} funding records")
else:
    print("No funding data found; training without funding rates.")

# Chronological split: 70% train / 15% val / 15% test
n = len(klines_df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = klines_df.iloc[:train_end].reset_index(drop=True)
val_df = klines_df.iloc[train_end:val_end].reset_index(drop=True)
test_df = klines_df.iloc[val_end:].reset_index(drop=True)

print(f"\nData split: train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}")

## Cell 3 — Create Environments

In [ ]:
from alphacluster.agent.config import TrainingConfig
from alphacluster.env.trading_env import TradingEnv
from stable_baselines3.common.vec_env import SubprocVecEnv, VecNormalize

config = TrainingConfig()

N_ENVS = config.n_envs  # 4 parallel environments

def make_train_env(rank: int):
    """Factory that returns a closure for SubprocVecEnv."""
    def _init():
        import os
        import warnings

        os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
        warnings.filterwarnings("ignore", category=DeprecationWarning)
        warnings.filterwarnings("ignore", message=".*Gym has been unmaintained.*")

        env = TradingEnv(
            df=train_df,
            funding_df=funding_df,
            window_size=config.window_size,
            episode_length=config.episode_length,
        )
        env.reset(seed=rank)
        return env
    return _init

print(f"Creating {N_ENVS} parallel training environments ...")
envs = SubprocVecEnv([make_train_env(i) for i in range(N_ENVS)])
train_env = VecNormalize(envs, norm_obs=False, norm_reward=True, clip_reward=10.0)

eval_env = None
if len(val_df) > config.window_size + config.episode_length:
    print("Creating validation environment ...")
    eval_env = TradingEnv(
        df=val_df,
        funding_df=funding_df,
        window_size=config.window_size,
        episode_length=config.episode_length,
    )
else:
    print("Validation set too small for evaluation; skipping eval callback.")

print(f"\nConfig: lr={config.learning_rate}, batch={config.batch_size}, "
      f"gamma={config.gamma}, ent_coef={config.ent_coef}")
print(f"Timesteps: {config.total_timesteps:,}")
print(f"Parallel envs: {N_ENVS} (with VecNormalize reward normalization)")
print(f"Curriculum learning: {'enabled' if config.curriculum_enabled else 'disabled'}")
print(f"Observation space: market=(576, 19), account=(12,)")
print(f"Action space: 3 directions x 4 sizes [2%,5%,10%,15%] x 3 leverages [5x,10x,15x] = 36")
print(f"Liquidation model: isolated margin (lose margin only, episode continues)")

## Cell 4 — Train

In [ ]:
import time
from alphacluster.agent.trainer import create_agent, save_agent, train
from stable_baselines3.common.callbacks import BaseCallback

# Save to Drive so models persist after runtime disconnects
MODELS_DIR = Path("/content/drive/MyDrive/alphacluster/models")
CHECKPOINT_DIR = MODELS_DIR / "checkpoints"


class ProgressCallback(BaseCallback):
    """Print a one-line progress update every ``log_interval`` timesteps."""

    def __init__(self, total_timesteps: int, log_interval: int = 50_000):
        super().__init__(verbose=0)
        self.total_timesteps = total_timesteps
        self.log_interval = log_interval
        self._start_time = None
        self._next_log = log_interval

    def _on_training_start(self) -> None:
        self._start_time = time.time()

    def _on_step(self) -> bool:
        if self.num_timesteps >= self._next_log:
            elapsed = time.time() - self._start_time
            pct = self.num_timesteps / self.total_timesteps * 100
            fps = self.num_timesteps / max(elapsed, 1)
            eta = (self.total_timesteps - self.num_timesteps) / max(fps, 1)
            print(
                f"  [{pct:5.1f}%] {self.num_timesteps:>10,} / {self.total_timesteps:,} "
                f"| {fps:.0f} fps | ETA {eta/60:.0f}m"
            )
            self._next_log += self.log_interval
        return True


print("Creating PPO agent (CNN+Transformer) ...")
agent = create_agent(train_env, config, verbose=0)

print(f"Training for {config.total_timesteps:,} timesteps ...")
print(f"Checkpoints saved to Drive: {CHECKPOINT_DIR}")
print(f"Curriculum phases: Learn Mechanics (0-30%) -> Introduce Costs (30-70%) -> Full Discipline (70-100%)\n")

progress_cb = ProgressCallback(config.total_timesteps, log_interval=50_000)

agent = train(
    agent=agent,
    config=config,
    eval_env=eval_env,
    checkpoint_dir=str(CHECKPOINT_DIR),
    run_tournament=False,
    progress_bar=False,
    verbose=0,
    extra_callbacks=[progress_cb],
)

final_path = MODELS_DIR / "ppo_trading_final"
save_agent(agent, final_path)
print(f"\nFinal model saved to {final_path}.pt")

## Cell 5 — Evaluate on Test Set

In [ ]:
from alphacluster.backtest.runner import run_backtest
from alphacluster.backtest.metrics import calculate_metrics, print_report

EVAL_SEED = 42
N_EVAL_EPISODES = 5

print(f"Running backtest on test set ({N_EVAL_EPISODES} episodes, seed={EVAL_SEED}) ...")
test_env = TradingEnv(
    df=test_df,
    funding_df=funding_df,
    window_size=config.window_size,
    episode_length=config.episode_length,
)

result = run_backtest(model=agent, env=test_env, n_episodes=N_EVAL_EPISODES, seed=EVAL_SEED)
metrics = calculate_metrics(result)
print_report(metrics)

## Cell 6 — Summary

In [ ]:
import os

print("=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print()
print(f"Model files on Google Drive: {MODELS_DIR}")
print()

for root, dirs, files in os.walk(str(MODELS_DIR)):
    level = root.replace(str(MODELS_DIR), "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = "  " * (level + 1)
    for f in sorted(files):
        size_mb = os.path.getsize(os.path.join(root, f)) / (1024 * 1024)
        print(f"{sub_indent}{f}  ({size_mb:.1f} MB)")

print()
print("Models persist on Drive. To use locally:")
print("  1. Download ppo_trading_final.pt from Drive: alphacluster/models/")
print("  2. Place it in your local models/ directory")
print("  3. Run: python scripts/evaluate.py --model-path models/ppo_trading_final.pt")